Day 3, Topic 5: Advanced Ufunc Features

## 📘 Day 3, Topic 5: Advanced Ufunc Features

### What This Topic Covers
This topic digs into three advanced ufunc capabilities that solve specific problems you'll encounter in real NumPy code:
1. **Type casting** — what happens when dtypes don't match
2. **`.at()` — unbuffered in-place operations** — solving the "duplicate index" problem
3. **`np.vectorize()` with `signature=`** — applying functions to sub-arrays generically

---

## 1. Type Casting in Ufuncs

### Automatic Upcasting
When ufuncs receive inputs of different dtypes, NumPy **promotes** to the highest-precision type:
```python
a = np.array([1, 2, 3], dtype=np.int32)
b = np.array([1.1, 2.2, 3.3], dtype=np.float64)
np.add(a, b).dtype   # → float64   (int32 promoted to float64)
```

### Controlling Output dtype with `out=`
Pre-allocating with a specific dtype forces the result to be cast into that type:
```python
out_arr = np.empty(3, dtype=np.float32)
np.add(a, b, out=out_arr)   # result stored as float32, not float64
```

### Integer Overflow — A Silent Danger
NumPy does **not** raise errors on integer overflow — it silently wraps around:
```python
arr = np.array([255, 255, 255], dtype=np.uint8)
np.add(arr, 1)   # → [0, 0, 0]  ← 255+1 wraps back to 0 for uint8!
```
`uint8` has range 0–255. Adding 1 to 255 overflows to 0. This affects both `np.add` and `np.add.at`.

> ⚠️ **Always check your dtypes** when working with fixed-width integers (int8, uint8, int16, etc.). Use a larger dtype or `np.clip()` if overflow is a concern.

---

## 2. `.at()` — Unbuffered In-Place Operations

### The Duplicate Index Problem
When you use fancy indexing to modify an array, **duplicate indices are only applied once**:
```python
a = np.zeros(5)
idx = np.array([1, 2, 1, 3])   # index 1 appears twice
a[idx] += 1
print(a)   # [0. 1. 1. 1. 0.]  ← index 1 only incremented once!
```
**Why?** NumPy buffers the reads before writing. Both `a[1] += 1` operations read the same old value `0.0`, then both write `1.0`. The second write doesn't "see" the first.

### The Fix: `np.add.at()`
`.at()` is **unbuffered** — each operation is applied immediately, one at a time:
```python
a = np.zeros(5)
idx = np.array([1, 2, 1, 3])
np.add.at(a, idx, 1)
print(a)   # [0. 2. 1. 1. 0.]  ← index 1 correctly incremented twice!
```

### Syntax
```python
np.ufunc.at(array, indices, values)
```
- `array` — modified in-place
- `indices` — can be 1D array, or tuple `(row_idx, col_idx)` for 2D
- `values` — scalar or array of values to apply

### Real-World Use: Building a Histogram Without `np.bincount`
```python
votes = np.random.randint(0, 5, size=10000)   # 10000 votes for 5 candidates
histogram = np.zeros(5, dtype=np.int32)
np.add.at(histogram, votes, 1)
# Each vote increments the right candidate's count — even if two votes hit same candidate
```

### 2D `.at()` with Coordinate Pairs
```python
arr = np.zeros((3, 4))
row_idx = np.array([1, 0, 2, 0])
col_idx = np.array([1, 3, 0, 1])   # (0,3) and (0,1) hit row 0 twice
np.add.at(arr, (row_idx, col_idx), 1)
# Correctly counts duplicates at (0,1) which appears twice
```

---

## 3. `np.vectorize()` with `signature=` — Sub-Array Functions

### The Problem: Functions That Take Arrays, Not Scalars
`np.vectorize()` without a signature applies a function **scalar-by-scalar**. But sometimes your function takes a whole row or column as input. The `signature=` parameter handles this:

```python
def row_mean(arr):
    return arr.mean(axis=-1)   # takes a 1D array, returns a scalar

# signature='(n)->()' means: takes a 1D array of length n, returns a scalar
row_mean_vec = np.vectorize(row_mean, signature='(n)->()')

data = np.random.rand(10, 5)
means = row_mean_vec(data)   # shape (10,) — one mean per row
```

### Signature Syntax
```
'(n)->()'     input: 1D of length n   → output: scalar
'(m,n)->()'   input: 2D matrix        → output: scalar
'(n)->(n)'    input: 1D               → output: 1D (same length)
'(m,n)->(m)'  input: 2D              → output: 1D (one value per row)
```

### Example: Row Range
```python
def row_range(arr):
    return np.max(arr) - np.min(arr)

range_func = np.vectorize(row_range, signature='(n)->()')
arr_2d = np.random.rand(3, 4)
range_func(arr_2d)   # → shape (3,) — one range value per row
```

> 💡 `np.vectorize` with `signature=` is great for **generic functions** that should apply to each row/column/block. For pure performance, writing the vectorized math directly (using axis= parameters) is faster — but `signature=` is cleaner for complex per-row logic.

In [1]:
import numpy as np

In [2]:
a = np.array([1, 2, 3], dtype=np.int32)
b = np.array([1.1, 2.2, 3.3], dtype=np.float64)
c = np.add(a, b)
print(c.dtype)

float64


In [3]:
out_arr = np.empty(3, dtype=np.float32)
np.add(a, b, out=out_arr)

array([2.1, 4.2, 6.3], dtype=float32)

In [4]:
#Problem with idx
a = np.zeros(5)
idx = np.array([1, 2, 1, 3])
a[idx] += 1
print(a)   # [0. 1. 1. 1. 0.]   Wait, index 1 only got incremented once!

[0. 1. 1. 1. 0.]


In [5]:
#Solved with at
a = np.zeros(5)
idx = np.array([1, 2, 1, 3])
np.add.at(a, idx, 1)
print(a)

[0. 2. 1. 1. 0.]


In [6]:
# Simulate 10000 votes for 5 candidates (0-4)
votes = np.random.randint(0, 5, size=10000)
histogram = np.zeros(5, dtype=np.int32)
np.add.at(histogram, votes, 1)
print(histogram)

[1944 2121 1947 1971 2017]


In [7]:
arr = np.zeros((3, 4))
row_idx = np.array([1, 0, 2, 0])
col_idx = np.array([1, 3, 0, 1])
np.add.at(arr, (row_idx, col_idx), 1)
print(arr)

[[0. 1. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]


In [20]:
#Custom Row‑Wise Mean Function
def row_mean(arr):
    return arr.mean(axis=-1)

row_mean_vec = np.vectorize(row_mean, signature='(n)->()')

data = np.random.rand(10, 5)
means = row_mean_vec(data)
print(means.shape)

(10,)


In [11]:
# Task 1: You have 100,000 random integers between 0 and 9.
# Use np.add.at to build a frequency histogram without loops.
arr = np.random.randint(0, 9, size=100_000)
histogram = np.zeros(9, dtype=np.int32)
np.add.at(histogram, arr, 1)
print(histogram)

[11164 11036 10965 11091 11244 11175 11063 11223 11039]


In [13]:
#Task 2: # Write a function row_range(arr) that returns max - min for each row.
# Use np.vectorize with signature to apply it to a 2D array.
def row_range(arr):
    return np.max(arr) - np.min(arr)

result = np.vectorize(row_range, signature='(n)->()')
arr_2d = np.random.rand(3, 4)
values = result(arr_2d)
print(values)


[0.54596836 0.32860457 0.70671542]


In [19]:
#Task 3: # Create a uint8 array of [255, 255, 255].
# Add 1 using np.add. What happens? (Overflow!)
# Now use np.add with casting='no'? Actually, use np.add.at to see overflow behavior.
arr = np.array([255, 255, 255], dtype=np.uint8)
result = np.add(arr, 1)
print(f"Original array: {arr}")
print(f"After adding one: {result}")
arr_at = np.array([255, 255, 255], dtype=np.uint8)
np.add.at(arr_at, [0, 1, 2], 1)
print(arr_at)

Original array: [255 255 255]
After adding one: [0 0 0]
[0 0 0]
